# Character-Level Bigram Language Model

This notebook is a personal reconstruction of [Andrej Karpathy](https://pt.wikipedia.org/wiki/Andrej_Karpathy)'s lesson, [The Spelled-Out Intro to Language Modeling: Building Makemore](https://www.youtube.com/watch?v=PaCmpygFfXo). In this lesson, he demonstrates how to build an autoregressive character-level bigram language model. It explores two approaches:

1. manually constructing probability distributions by counting bigrams;
2. discovering these distributions using backpropagation and gradient descent.

The purpose of this notebook is for my own self-learning, and shouldn't add much to the original lesson beyond extra verbosity. Parts of the notebook, namely the code snippets, may have been copied from the source lesson verbatim.

## Setup

First we need to download a dataset of names that will be used to train our model:

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/makemore/master/names.txt

Once downloaded, we can read the names from the file into a list:

In [ ]:
words = open('names.txt', 'r').read().splitlines()
words[:10]

Let's check some basic statistics, including the total number of names, and the minimum and maximum name lengths:

In [ ]:
len(words), min([len(w) for w in words]), max([len(w) for w in words])

## Manually building the model

To create new strings that follow patterns from a dataset, we can track how often one letter follows another, build probability distributions representing these patterns, and use them to generate new strings by sampling from them, one character at a time. We start with the first character, sample the next based on the first, then repeat until we hit an *End-of-Sequence (EOS) character*, which signals the end of the string.

This is called a **Character-level Bigram Language Model**, and you can think of it as a [Markov Chain](https://en.wikipedia.org/wiki/Markov_chain), where each character represents a state, and each transition to another character has a certain probability, with *EOS* being an [absorbing state](https://en.wikipedia.org/wiki/Absorbing_Markov_chain).


Let's get started! First we need to generate a dictionary to count the frequency of each character-level bigram (pairs of consecutive characters) in the dataset:

In [ ]:
EOS_CHAR = '.' # Special character used to represent the end of a string
bigrams_map = {}
for word in words:
  # Wrap each word with a EOS character:
  # - the one at the start can be thought of as the end of the previous generation and serves as a neutral token from where to sample the first token
  # - the one and the end signals that there are no more characters in that name
  chars = [EOS_CHAR] + list(word) + [EOS_CHAR]
  for char1, char2 in zip(chars, chars[1:]): # Read two characters at a time (eg: '.alf.' becomes [('.', 'a'), ('a', 'l'), ('l', 'f'), ('f', '.')])
    bigram = (char1, char2) # A bigram is a sequence of two adjacent characters
    bigrams_map[bigram] = bigrams_map.get(bigram, 0) + 1 # Increment the count for the presence of this bigram in the dataset
bigrams_map

To better understand our dataset, let's look at the $10$ most frequent bigrams:

In [ ]:
sorted(bigrams_map.items(), key = lambda x: -x[1])[:10]

From the sorted list above we can already tell that:

- Most names end with `n`;
- Most names start with `a` or `k`;
- `an` is the most common bigram if we disregard `.` (EOS character).

To build our model, we'll first need to convert each character into a unique number that represents it. This numerical representation will make it easier for our model to work with the data:

In [ ]:
chars = sorted(list(set(''.join(words))))
stoi_map = {s:i+1 for i, s in enumerate(chars)}
stoi_map['.'] = 0
stoi_map

With our characters converted to numbers, we can represent the bigram frequencies as a PyTorch tensor of shape $(27, 27)$, where each row corresponds to a character, each column corresponds to the next character, and each cell holds the count of how often that specific bigram appears in the dataset:

In [ ]:
import torch

num_chars = len(stoi_map) # Number of chars = 27
N = torch.zeros((num_chars, num_chars), dtype=torch.int32) # Create a tensor of shape (27, 27) of int32 data type
for bigram, frequency in bigrams_map.items():
  char1, char2 = bigram # Unpack the bigram
  char1_i, char2_i = stoi_map[char1], stoi_map[char2] # Convert bigram chars to ints
  N[char1_i, char2_i] = frequency # Assign the count to the tensor position for that bigram
N.shape, N

Tensor outputs can be tricky to interpret, so let's make it easier by visualizing the data as a heatmap. This will give us a clearer picture of the patterns and relationships between characters in our bigram counts:

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(N)

In the heatmap, we can see lighter colors concentrated in the $X = 0$ column and the $Y = 0$ row. This happens because `0` represents the `.` (the *EOS character*), which appears at the start and end of every word, so there's naturally more activity in those areas. You'll also spot some darker pixels—these represent characters that are less likely to appear at the beginning or end of a name.

It's hard to tell which specific bigrams these pixels correspond to, so let's make the heatmap more informative by adding annotations to each cell. We'll label each cell with the corresponding bigram and its frequency to give us a clearer understanding of the data:

In [ ]:
itos_map = {i:s for s,i in stoi_map.items()} # Create a reverse lookup table allowing us to convert integers to corresponding chars

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(num_chars):
  for j in range(num_chars):
    bigram = itos_map[i] + itos_map[j]
    bigram_frequency = N[i, j].item()
    plt.text(j, i, bigram, ha="center", va="bottom", color="gray")
    plt.text(j, i, bigram_frequency, ha="center", va="top", color="gray")
plt.axis('off')

Each row in our tensor `N` represents the first character, and the columns represent how frequently each possible next character appears:

In [ ]:
N[0] # Frequency of characters following '.'

If we normalize these rows, we can interpret them as probability distributions:

In [ ]:
N[0] / torch.sum(N[0]) # Probability of characters following '.'

If you have a probability distribution represented as an array, where each value indicates the probability of sampling that specific position, you can use `torch.multinomial` to sample from it:

In [ ]:
import torch

# Initialize a generator with a fixed seed so that make this cell always have the same output
SEED = 42; generator = torch.Generator().manual_seed(SEED)

# The probability distribution: all these values sum to 1.0
probability_distribution = torch.tensor([0.5, 0.2, 0.3, 0.1])

# Draw N samples from the distribution
sampled_values = torch.multinomial(
    probability_distribution, # Sample from this probability distribution
    num_samples=50, # Draw N samples from the distribution
    replacement=True, # Make it possible to draw the same value multiple times
    generator=generator # Use manually initialized RNG so that this cell always returns the same output
)

# Count how many times each number was sampled
sampled_counts = torch.bincount(sampled_values, minlength=len(probability_distribution))

# Calculate the sampled distribution
sampled_distribution = sampled_counts / sampled_counts.sum()

{
    "probability_distribution" : probability_distribution,
    "sampled_values" : sampled_values,
    "sampled_counts" : sampled_counts,
    "sampled_distribution" : sampled_distribution
}

As you can see from the output above, the numbers were indeed sampled in proportion to their probability distribution. As you increase `num_samples`, the `sampled_distribution` should get closer to the `probability_distribution`, further confirming the reliability of the sampling.

With this knowledge, we can now write the code used to sample the next most frequent letter after `.` (the *EOS character*, also used to annotate the start of a word):

In [ ]:
N0_P = N[0] / torch.sum(N[0])
index = torch.multinomial(N0_P, num_samples=1)
index = index.item() # torch.multinomial returns a tensor, need to unpack its item to a scalar
next_char = itos_map[index] # convert index back to character
next_char

Since we're not providing a manually instantiated generator to `torch.multinomial()`, the next character may be different every time you run that cell, so feel free to run it a couple of times.

Now that we've covered how to turn a row of character occurrence frequencies into a probability distribution, let's convert the entire bigram frequency tensor `N`, into a bigram probability tensor `P`, so that given any character, by retrieving its row, we'll get the probability distribution of the character that comes after it.

To create the bigram probability tensor `P`, we need to know how to sum up each row, collapsing the row into single value representing its sum, turning the tensor from shape $(27, 27)$ to $(27, 1)$. Let's figure out to how to do that on a small tensor first:

In [ ]:
a = torch.tensor([[1, 2], [3, 4]])
a.shape, a

We now have a 2D tensor of shape $(2, 2)$, let's run `.sum()` on it:

In [ ]:
a.sum()

We got the sum of all values in the tensor: $1 + 2 + 3 + 4 = 10$, which is not what we want.

Let's specify that we want to sum the row dimension `0`:

In [ ]:
a.sum(dim=0)

Also not what we want, it seems we summed up the columns instead... This happened because `a.sum(dim=0)` actually means that the sum runs along the rows dimension: it starts in row $0$ with value $1$, then row $1$ with value $3$, giving us $1 + 3 = 4$, the first value in our resulting `tensor([4, 6])`.

What we really want is to perform the sum along the columns dimension $1$, collapsing that dimension into a single value representing the sum for all the columns in that row:

In [ ]:
a.sum(dim=1), a.sum(dim=1).shape

Now we got the expected `tensor([3, 7])`, but an unexpected shape $(2)$. This happened because the sum operation performs a *squeeze* by default, getting rid of any resulting dimension with value $1$.

Let's run it again while disabling the default squeezing behavior by passing the parameter `keepdim=True`:

In [ ]:
a.sum(dim=1, keepdim=True), a.sum(dim=1, keepdim=True).shape

Perfect! We can now run this on our target bigram frequency tensor `N` to create a bigram probability tensor `P`:

In [ ]:
P = N.float()
P /= P.sum(1, keepdim=True) # NOTE: in-place operations are more efficient because they don't allocate a new tensor
P.shape, P

Let's just perform a sanity check by verifying that all the rows add up to $1.0$ before proceeding (remember that each row represents a probability distribution):

In [ ]:
P.sum(dim=1).shape, P.sum(dim=1)

All good! Another validation we can make is to ensure that our model assigns a higher probability to all bigrams found in the dataset than that of random chance. In this particular case, random chance can be determined to be a probability that is lower than $1 / 27$ (any character out of all $27$ possible characters):

In [ ]:
learned_count = 0 # Number of bigrams whose assigned probability is higher than random chance
bigram_count = 0 # Number or bigrams found in the dataset (includes repetitions)
for word in words:
  for char1, char2 in zip(word, word[1:]):
    char1_i, char2_i = stoi_map[char1], stoi_map[char2]
    bigram_probability = P[char1_i, char2_i] # Retrieve bigram probability from model
    learned_count += 1 if bigram_probability > 1/27 else 0 # Count as a bigram that was learned when its assigned probability is above random chance
    bigram_count += 1 # Number of bigrams counted so far
learned_percentage = learned_count / bigram_count # Calculate percentage of bigrams whose probability is above random chance
learned_percentage

Our model assigns a probability higher than chance to $78\%$ of the bigrams present in the dataset, so we're pretty confident our model has learned something and will not just output random noise.

We can finally write a function that uses our model to sample new words:

In [ ]:
def sample_word(generator=None, uniform=False):
  current_char_index = 0 # Start with character '.'
  sampled = []
  while True:
    next_char_probs = P[current_char_index] # Retrieve probability distribution of next character
    if uniform: next_char_index = torch.randint(0, len(next_char_probs), (1,), generator=generator) # Sample next char from uniform distribution (demo purposes)
    else: next_char_index = torch.multinomial(next_char_probs, num_samples=1, generator=generator) # Sample index of next character using previous char's probability distribution (default)
    next_char_index = next_char_index.item() # Unpack tensor value into scalar
    next_char = itos_map[next_char_index] # Map next char index into respective char using previously created lookup table
    sampled.append(next_char) # Add to list of sampled chars for current word
    if next_char_index == 0: break # In case we have a sampled an EOS character then stop sampling (word is complete)
    current_char_index = next_char_index # Otherwise next char index is now the current char (autoregressive sampling)
  word = "".join(sampled) # Join all collected chars into a single string
  return word

generator = torch.Generator().manual_seed(SEED) # Create a manually initialized generator to make this cell always have the same output
for x in range(20): print(sample_word(generator)) # Sample and print 20 words

The results aren't great, but they occasionally sound like real names, which means we're on the right track. The issue likely comes from our model's approach—predicting the next character based only on the previous one isn't strong enough (for example, using the last $N$ characters could yield better results).

Let's sample words from a uniform distribution to see how our model stacks up in comparison:

In [ ]:
for x in range(20): print(sample_word(generator, uniform=True))

We sampled complete gibberish now, so we can rest assured that our model is not that bad.

## Training a model instead

We created the previous model by manually counting the frequencies of each character-level bigram, using that to calculate the probability distributions of those occurrences, and using them to sample new words. But if we were to solve this issue in a true machine learning fashion, we would have the model learn the probability distributions from scratch. Let's change our approach to solve the problem that way!

First and foremost, we need a loss function to guide us. The loss function will tell us how good the model is at any given moment, allowing us to calculate how each parameter contributed to the current loss value using *backpropagation*, and to adjust the parameters in order to minimize the loss function using *gradient descent*, guiding us towards a better model with each iteration.

Our goal is to maximize the [likelihood](https://en.wikipedia.org/wiki/Likelihood_function) that the model correctly predicts the sequence of characters in each word. The likelihood is calculated by multiplying the probabilities of each bigram in a word.

Multiplying probabilities can however lead to extremely small values, which may not be representable in floating-point format, making the resulting value be incorrectly stored in memory as $0$. To avoid this, we use the logarithm of the probabilities to calculate the **log likelihood** instead. We do this because logarithms have the useful property of turning small numbers into large ones. Let's plot the logarithm function to see it for ourselves:

In [ ]:
import numpy as np
x = np.linspace(0.1, 20, 400)
y = np.log(x)
plt.plot(x, y)
plt.grid(True)

As you can see above, as $x$ gets closer to $0$, $y$ grows in magnitude towards $-\infty$. This solves the problem of the small numbers, but leaves us with the problem of resulting negative numbers, which are not ameanable to *gradient descent*, where the objective is to decrease the loss value towards $0$. To solve that problem, we can just negate the logarithm and use the **negative log likelihood** instead.

Now that we have a metric that satisfies all the requirements of a good loss function, let's implement it:

In [ ]:
def calc_log_likelihood(word, verbose=False):
  log_likelihood = 0.0
  for char1, char2 in zip(word, word[1:]):
    bigram = (char1, char2)
    char1_i, char2_i = stoi_map[char1], stoi_map[char2]
    prob = P[char1_i, char2_i] # Retrieve the probability of char2 appearing after char1
    log_prob = torch.log(prob) # Apply logarithm to turn probability into a bigger value
    log_likelihood += log_prob # Add to the total value because log(a*b) = log(a) + log(b)
    if verbose: print(f"{char1}{char2}: prob: {prob:.4f}, log prob: {log_prob:.4f}")
  return log_likelihood

for word in words[:3]:
  log_likelihood = calc_log_likelihood(word, verbose=True)
  negative_log_likelihood = -log_likelihood
  print(f"word={word} (log_likelihood={log_likelihood}; negative_log_likelihood={negative_log_likelihood})")

Let's try an edge case by checking the likelihood for an unusual name:

In [ ]:
calc_log_likelihood("andrejz", verbose=True)

We're getting $-\infty$ for the log likelihood of `jz` because its probability is $0.0$ (no ocurrence in dataset), and because $log(0) = -\infty$. But since `jz` is a valid bigram, we shouldn't get this.

To fix this issue, we can add a small value to the original values and only then calculate the probabilities:

In [ ]:
P = (N + 0.001).float()
P /= P.sum(1, keepdim=True)
calc_log_likelihood("andrejz", verbose=True)

Since no probability is $0.0$ anymore, there are no more negative infinities, so that problem is now solved.

Let's now to create a [training set](https://en.wikipedia.org/wiki/Training,_validation,_and_test_data_sets) to train our model with. In this *training set* each $x$ value is a character and each $y$ value is the next character found in the data:

In [ ]:
def create_dataset(words):
  xx, yy = [], []

  for word in words:
    chars = [EOS_CHAR] + list(word) + [EOS_CHAR]
    for char1, char2 in zip(chars, chars[1:]):
      char1_i = stoi_map[char1]
      char2_i = stoi_map[char2]
      xx.append(char1_i)
      yy.append(char2_i)

  xx = torch.tensor(xx)
  yy = torch.tensor(yy)
  return xx, yy

xx, yy = create_dataset(words)
xx.shape, xx, yy.shape, yy

To recap, our dataset is a composed of:
- $X$ (inputs): a tensor of shape $(228146)$, representing $228146$ characters, where each character can be one out of $27$ possible characters and is encoded as a value in range $[0,26]$.
- $Y$ (outputs): exactly the same format as the inputs, but representing the character that was found in the corresponding input at $X$.

To feed the inputs into the neural network, we should [one-hot](https://en.wikipedia.org/wiki/One-hot) encode them first. This process converts each character into a vector where all elements are $0$ except for a single element with value $1$, representing the specific character.

The main reason for this encoding is to avoid introducing unintended relationships between characters. If we used simple integers (e.g., $a = 1$, $b = 2$), the network might mistakenly assume that characters with higher numbers are more important or related.

One-hot encoding ensures that each character is represented equally, with no implicit order or magnitude. Every character starts with the same "neutral" signal, with the model's weights then determining how these signals are adjusted.

Let's one-hot encode the inputs then:

In [ ]:
import torch.nn.functional as F
xx_onehot = F.one_hot(xx, num_classes=num_chars).float() # F.one_hot() returns tensors of type int32 by default
xx_onehot.shape, xx_onehot

Let's look closer to one of the inputs:

In [ ]:
xx[20], itos_map[xx[20].item()], xx_onehot[20]

We retrieved the character represented in the $20th$ input and determined that it is character `b`, represented by index `2`, which means its one-hot vector has `27` positions (there are `27` possible characters), but only the position at index `2` has a value set to `1`.

Now let's go the other way around and decode the one-hot vector back to the character to reinforce this concept:

In [ ]:
char_onehot = xx_onehot[20] # Retrieve the 20th input (one-hot vector)
char_i = char_onehot.nonzero()[0].item() # Retrieve the index of the first position with a non-zero value
itos_map[char_i], char_i, char_onehot

Let's visualize a slice of the `xx_onehot` tensor. Each row will have a yellow dot indicating where the $1$ is placed, showing which character is represented in that row:

In [ ]:
plt.imshow(xx_onehot[:20,:])

Let's also one-hot encode the $Y$ values:

In [ ]:
yy_onehot = F.one_hot(yy, num_classes=num_chars).float()
yy_onehot.shape, yy_onehot

Time to focus create the neural network itself. We want to create a simple network similar to a [Perceptron](https://en.wikipedia.org/wiki/Perceptron). This network will have a single layer of neurons, with the activations of these neurons serving as the model's output.

Since we're predicting the values in the $Y$ tensor, which contains one-hot vectors of length $27$, we need $27$ neurons in this layer. Each neuron will output a value, and the index with the highest activation will correspond to the predicted character.

Given that the inputs from the $X$ tensor are also one-hot vectors of length $27$, each neuron will have $27$ weights attached to it—one for each input value.

Let's create the tensor representing those weights:

In [ ]:
generator = torch.Generator().manual_seed(SEED) # Use generator to make sure we always get the same weights when we run this cell
W = torch.randn((num_chars, num_chars), generator=generator, requires_grad=True) # Enable `requires_grad` so that gradients are tracked during calculations
W.shape

If we visualize the weights in their initial state we see just noise due to their random initialization:

In [ ]:
plt.imshow(W.detach().numpy()) # You need to convert the tensor to numpy array first because `imshow()` doesnt support tensors with gradient tracking

To perform the forward pass, we only need to calculate the [matrix multiplication](https://en.wikipedia.org/wiki/Matrix_multiplication) $X \times W$, this will give us the activation for all the neurons.

The outputs of the final layer of a neural network, particularly in classification tasks, are known as **logits** before they are passed through a softmax or another activation function to convert them into probabilities.

Due to the simplicity of this particular problem, we won't add any bias, or apply a non-linear function to the neurons' output:

In [ ]:
logits = xx_onehot @ W # @ is the operator for matrix multiplication (matmul)
xx_onehot.shape, W.shape, logits.shape, logits

By performing the matrix multiplication between the input tensor $X$ of shape $(228146, 27)$ and the weights tensor $W$ of shape $(27, 27)$, we obtain a tensor of shape $(228146, 27)$.

This shape outcome occurs because, in matrix multiplication, multiplying a matrix of shape $(m \times n)$ by one of shape $(n \times p)$ results in a matrix of shape $(m \times p)$.

This result represents the predictions for each possible next character, where each prediction is a one-hot vector of length $27$, corresponding to the $228146$ inputs.

Let's examine the *logits* for the first input:

In [ ]:
logits[0], torch.sum(logits[0])

We'll need to convert the *logits* into a valid probability distribution (all values in the range $[0, 1]$ and summing to $1.0$), representing the probability of each character ocurring after the one provided in the respective input.

First, we make all values positive by exponentiating them, which not only ensures positivity but also emphasizes the most significant activations. This approach is better than using the absolute value, which would make the value become positive, but the $abs()$ operation only provides the gradients $-1$ or $1$, whereas $exp()$ provides a continuous, input-dependent gradient, improving the efficiency of *gradient descent*.

After obtaining positive values, we can then normalize them to form a probability distribution that aligns with the $Y$ tensors from our training set, which are within $[0, 1]$.

Let's convert the *logits* to probability distributions:

In [ ]:
counts = logits.exp() # By forcing the network to express the numbers as positive, when it learns the right distributions these values will represent the bigram frequencies
probs = counts / counts.sum(dim=1, keepdims=True) # Normalize the output probabilities
probs.shape, probs, probs.sum(dim=1)

Perfect, our model's output can now be read as the probability of each character being found after the input's character.

We now want to use the outputs to calculate the loss, let's first look at our input/output pairs again:

In [ ]:
xx, yy

We want to find the probability assigned by the model for specific character sequences. For example, to find the probability of character `13` appearing after character `5`, we would check `probs[1, 13]`. This is because we're focusing on the probabilities associated with the input at index `1` (`xx[1]` corresponds to character `5`), and the probability of the output character being `13`, remembering the outputs are one-hot encoded vectors of size `27`, where each index represents a character, meaning that index `13` represents character `13`.

Now, let's examine the probabilities for the first three input/output pairs:

In [ ]:
{
    f"{itos_map[xx[0].item()]}{itos_map[yy[0].item()]}" : probs[0, 5],  # 5 is the value in yy[0]
    f"{itos_map[xx[1].item()]}{itos_map[yy[1].item()]}" : probs[1, 13], # 13 is the value in yy[1]
    f"{itos_map[xx[2].item()]}{itos_map[yy[2].item()]}" : probs[2, 13]  # 13 is the value in yy[2]
}

Well, these probabilities suck... pretty normal though, because we haven't trained our model yet. You can check back on these probabilities again after training.

You can access the same probabilities with a single lookup operation, since PyTorch tensors allow indexing multiple values across multiple dimensions. For example, if we wanted a single tensor with the probabilities above, we could just feed $[0, 1, 3]$ as the first dimension, and $[5, 13, 13]$ as the second one:

In [ ]:
probs[[0, 1, 3], [5, 13, 13]]

We got the exact same probabilities, just as expected. You could also access the same values by providing the respective dataset slices:

In [ ]:
probs[x[:3], yy[:3]]

Also worked! We can now see a clear path towards calculating our **negative log likelihood** loss function with a single call. We need to collect all the probabilities for the batch in question (in this case our entire dataset), calculate the logarithm for each of them, then calculate their mean.

One step at a time, first let's collect the probabilities of each character from $X$ being succeeded by each corresponding character from $Y$:

In [ ]:
probs[xx, yy].shape, probs[xx, yy]

Now let's get the logarithm for those values (to make the values bigger):

In [ ]:
probs[xx, yy].log()

We can now get their mean because $log(a \times b) = log(a) + log(b)$. We want to multiply the probabilities of each bigram, but since we converted them to logarithms, we can now just sum them to get the *log likelihood loss*:

In [ ]:
probs[xx, yy].log().mean()

And now we just need to negate the value to get the *negative likelihood loss*, our desired loss function:

In [ ]:
nll_loss = -probs[xx, yy].log().mean()
nll_loss

We can calculate this very same loss function more easily by using PyTorch utils. First we use `log_softmax()` to convert the logits to a probability distribution, by applying softmax on dimension $1$ and then getting the logarithms for those values (aka *log probabilities*):

In [ ]:
log_probs = F.log_softmax(logits, dim=1)
log_probs

With the *log probabilities* in hand, we can now calculate the *negative log-likelihood loss* using `F.nll_loss()`. This function measures how well the predicted *log probabilities* align with the actual target labels $Y$:

In [ ]:
nll_loss = F.nll_loss(log_probs, yy)
nll_loss

In our manual calculation we got $3.7686$, but here we got $3.6966$. This can be attributed to the superior efficacy of using the PyTorch utils, which fuse the operations resulting in higher numeric stability, and therefore less precision loss across the calculations (for more info, look into the [Log-Sum-Exp trick](https://gregorygundersen.com/blog/2020/02/09/log-sum-exp/)):

You can simplify the loss calculation even further by just using `F.cross_entropy()`, which will do pretty much the same thing as running `F.log_sofmax()` followed by `F.nll_loss()`:

In [ ]:
loss = F.cross_entropy(logits, yy)
loss

With this knowledge, we'll just use `F.cross_entropy(logits, yy)` going forward.

Before starting with the training, let's first note down the loss we got from the model we manually created before. We'll know the model we're about to train as reached its goal when its loss is close to, or below that loss value:

In [ ]:
target_loss = -P[xx, yy].log().mean()
target_loss

Ok, so we know we want to train a model whose loss is close to $2.4540$.

Now let's write the training code and run it for a couple of epochs:

In [ ]:
def train(n_epochs, learning_rate=0.01, verbose=False):
  # Convert inputs to one-hot encoded vectors
  xx_enc = F.one_hot(xx, num_classes=num_chars).float()

  # Run gradient descent optimization for N epochs
  xx_length = len(xx)
  for epoch in range(n_epochs):
    # Forward pass
    logits = xx_enc @ W

    # Calculate the loss
    loss = F.cross_entropy(logits, yy)

    # Print the status
    if verbose: print(f"epoch:{epoch};loss={loss}")

    # Perform backpropagation to calculate the gradients
    W.grad = None # Reset gradients first, otherwise it will accumulate over values from last iteration
    loss.backward() # Perform backpropagation

    # Perform gradient descent step by deducting gradients from the data
    W.data -= learning_rate * W.grad
  return loss

# Run training for N epochs and output the resulting loss
train(10, learning_rate=0.01, verbose=True)

Loss is steadily decreasing, let's bump up the learning rate by an order of magnitude:

In [ ]:
train(10, learning_rate=0.1, verbose=True)

Still decreasing steadily, let's bump it up again:

In [ ]:
train(10, learning_rate=1, verbose=True)

Still steady, again:

In [ ]:
train(10, learning_rate=10, verbose=True)

Another bump, this time smaller, and let's train longer:

In [ ]:
train(1000, learning_rate=50)

Loss is close enough to our target of $2.454$ so we're good!

Let's try sampling from our trained model and see what we get:

In [ ]:
def sample_word(generator=None, uniform=False):
  current_char_index = 0 # Start with character '.'
  sampled = []
  while True:
    # Perform forward pass
    xx_enc = F.one_hot(torch.tensor([current_char_index]), num_classes=num_chars).float()
    logits = xx_enc @ W
    counts = logits.exp()
    next_char_probs = counts / counts.sum(1, keepdims=True)

    # Sample next character
    next_char_index = torch.multinomial(next_char_probs, num_samples=1, generator=generator) # Sample index of next character using previous char's probability distribution (default)
    next_char_index = next_char_index.item() # Unpack tensor value into scalar
    next_char = itos_map[next_char_index] # Map next char index into respective char using previously created lookup table
    sampled.append(next_char) # Add to list of sampled chars for current word
    if next_char_index == 0: break # In case we have a sampled an EOS character then stop sampling (word is complete)
    current_char_index = next_char_index # Otherwise next char index is now the current char (autoregressive sampling)

  # Join all collected chars into a single string
  word = "".join(sampled)
  return word

generator = torch.Generator().manual_seed(SEED) # Create a manually initialized generator to make this cell always have the same output
for x in range(20): print(sample_word(generator)) # Sample and print 20 words

We sampled pretty much the same words as from the model we had manually created before. This is to be expected, because that model was optimal, and we have achieved a similar loss.

It's worth investigating what the trained weight tensor `W` actually learned. Let's plot its histogram:

In [ ]:
plt.imshow(W.detach().numpy())

The histogram plot of the model's weights is not noisy anymore, there are some noticeable patterns in there. Interestingly, there's a particularly dark pixel in the top left corner, which stands out from the rest. This position corresponds to the bigram `..`, which never occurs, resulting in a zero value, hence the darker color.

Let's plot it alongside the probabilities `P` from our previous manually created model:

In [ ]:
# Plotting the weight matrices side by side
plt.figure(figsize=(10, 4))

# Plot for W
plt.subplot(1, 2, 1)
plt.imshow(W.detach().numpy(), cmap='viridis')
plt.colorbar()
plt.title("Weight Tensor W")

# Plot for P
plt.subplot(1, 2, 2)
plt.imshow(P.detach().numpy(), cmap='viridis')
plt.colorbar()
plt.title("Bigram Probabilities P")

Hmmm, those are eerily similar... notice how they are displaying the same arrangement but in a different scale, let's normalize `W` to the same scale:

In [ ]:
# Plotting the weight matrices side by side
plt.figure(figsize=(10, 4))

# Normalize W
WN = W.exp()
WN = WN / WN.sum(1, keepdims=True)

# Plot for W
plt.subplot(1, 2, 1)
plt.imshow(WN.detach().numpy(), cmap='viridis')
plt.colorbar()
plt.title("Normalized Weight Tensor W")

# Plot for P
plt.subplot(1, 2, 2)
plt.imshow(P.detach().numpy(), cmap='viridis')
plt.colorbar()
plt.title("Bigram Probabilities P")

Voila! Our trained weight tensor `W` learned the same model as our manually created model encoded in the tensor `P`!

Given that we're using one-hot encoded inputs and our model has no bias or non-linear activations, the weights learned a distribution that mirrors the distribution found in the bigram counts, that's why normalizing them yields an extremely similar plot.

## THE END

Check my repo for more **AI/ML** notebooks: https://github.com/tsilva/aiml-notebooks